In [0]:
#/Volumes/workspace/amazon_data/amazon/flipkart.csv


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan,when,count, avg, expr,sum

spark = SparkSession.builder.appName('flipkart_dataset').getOrCreate()


In [0]:
file_path = '/Volumes/workspace/amazon_data/amazon/flipkart.csv'

fk_df = spark.read.csv(file_path, header=True, inferSchema = True)
fk_df.display()

In [0]:
fk_df.printSchema()

In [0]:
fk_df.select([count(when(col(c).isNull(), c)).alias(c) for c in fk_df.columns]).display()

In [0]:
fk_df_clean = fk_df.dropna()
fk_df_fill = fk_df.fillna({"Rating":0}).dropna()


In [0]:
fk_df_tfx = fk_df.withColumn("Effective_Price", expr("Price-(Price*Discount/100)")) 

fk_df_tfx.select("ProductName","Price","Discount","Effective_Price").display(5)

In [0]:
high_rated = fk_df_fill.filter(col("Rating")>4).display()

In [0]:
#group the category cal avg rating
avg_rating_df = fk_df_fill.groupBy("maincateg").avg("Rating")

avg_rating_df.display()


In [0]:
#total revenue by categ

total_Rev= fk_df_fill.groupBy("maincateg").agg(sum("Rating"))
total_Rev.display()

In [0]:
#Save the Processed data

output_table = 'FlipKart_Analysis'

fk_df_fill.write.mode('overwrite').saveAsTable(output_table)

In [0]:
%sql

select * from Flipkart_Analysis limit 20